# Parametric VaR

The Parametric (Variance-Covariance) approach assumes that portfolio returns are
**normally distributed**. Instead of sorting historical returns, we estimate the
mean and standard deviation of the return distribution and use the **z-score**
at the desired confidence level to compute VaR analytically.

## 1. Imports

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy import stats
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Set up parameters
TICKER = 'GOOG'
CONFIDENCE_LEVEL = 0.99
LOOKBACK_DAYS = 251  # ~1 year of trading days
HORIZON_DAYS = 10
PORTFOLIO_VALUE = 1_000_000

print(f"Analyzing {TICKER} with {CONFIDENCE_LEVEL:.0%} confidence")
print(f"Lookback period: {LOOKBACK_DAYS} days, Horizon: {HORIZON_DAYS} days")
print(f"Portfolio value: ${PORTFOLIO_VALUE:,}")

Analyzing GOOG with 99% confidence
Lookback period: 251 days, Horizon: 10 days
Portfolio value: $1,000,000


## 2. Fetch Prices

In [3]:
def fetch_prices(ticker, lookback, var_date=None):
    """Fetch daily close prices for a ticker.

    Gets the last `lookback` trading days of data up to the day before `var_date`.
    If `var_date` is not given, it uses the last business day.
    """
    if var_date is None:
        var_date = (pd.Timestamp.today() - pd.offsets.BDay()).date()

    calendar_days = int(lookback * 1.6)
    start = var_date - pd.Timedelta(days=calendar_days)

    df = yf.download(
        ticker,
        start=start.strftime("%Y-%m-%d"),
        end=var_date.strftime("%Y-%m-%d"),
        progress=False,
        interval="1d",
        auto_adjust=True
    )

    if df.empty:
        raise ValueError(f"No data returned for ticker '{ticker}'.")

    prices = df["Close"].squeeze()
    prices.name = ticker
    result = prices.tail(lookback)
    return result

In [4]:
prices = fetch_prices(TICKER, LOOKBACK_DAYS)
print(f"Shape: {prices.shape}")
print(f"Start date: {prices.index.min().date()}")
print(f"End date: {prices.index.max().date()}")

Shape: (251,)
Start date: 2025-03-27
End date: 2026-03-26


## 3. Return Calculation

Calculate daily returns from the price data.

In [5]:
def compute_returns(prices, kind="arithmetic"):
    """Compute daily returns from a price series.

    kind : "arithmetic" or "log"
        arithmetic  ->  (P_t - P_{t-1}) / P_{t-1}
        log         ->  log(P_t) - log(P_{t-1})
    """
    if kind == "log":
        returns = np.log(prices) - np.log(prices.shift(1))
        returns.name = "Daily Log Return"
    else:
        returns = (prices - prices.shift(1)) / prices.shift(1)
        returns.name = "Daily Return"
    return returns.dropna()

In [6]:
daily_returns = compute_returns(prices, kind="log")

print("Daily log return series:")
print("Shape: ", daily_returns.shape)
print("\n ", daily_returns.head())

Daily log return series:
Shape:  (250,)

  Date
2025-03-28   -0.050114
2025-03-31    0.001089
2025-04-01    0.016820
2025-04-02   -0.000126
2025-04-03   -0.040007
Name: Daily Log Return, dtype: float64


## 4. Estimate Distribution Parameters

The parametric method models daily returns as $R \sim \mathcal{N}(\mu, \sigma^2)$.  
We estimate the **mean** ($\mu$) and **standard deviation** ($\sigma$) from the
historical sample, then derive VaR from the normal quantile.

In [7]:
def estimate_distribution(returns):
    """Estimate mean and standard deviation of daily returns."""
    mu = returns.mean()
    # ddof=1 ensures we divide by (N-1), giving an unbiased estimate of volatility from sample data
    sigma = returns.std(ddof=1)
    return mu, sigma

In [8]:
mu, sigma = estimate_distribution(daily_returns)

print(f"Mean daily return (mu):      {mu:.6f} ({mu*100:.4f}%)")
print(f"Std dev daily return (sigma): {sigma:.6f} ({sigma*100:.4f}%)")

Mean daily return (mu):      0.002162 (0.2162%)
Std dev daily return (sigma): 0.018841 (1.8841%)


## 5. Calculate VaR and ES

**Parametric VaR** at confidence level $c$ is:

$$\text{VaR}_{1\text{-day}} = -(\mu + z_{\alpha}\,\sigma)$$

where $z_{\alpha} = \Phi^{-1}(1 - c)$ is the z-score for the lower tail.

**Expected Shortfall (ES)** under the normal assumption is:

$$\text{ES}_{1\text{-day}} = -(\mu - \sigma \cdot \frac{\phi(z_{\alpha})}{1-c})$$

where $\phi$ is the standard normal PDF.

In [9]:
def calculate_parametric_var(returns, confidence):
    """Compute parametric VaR assuming normally distributed returns.

    VaR = -(mu - z * sigma)   where z = norm.ppf(confidence).
    Returns VaR as a positive loss fraction.
    """
    mu, sigma = estimate_distribution(returns)
    z = float(stats.norm.ppf(confidence))
    return -(mu - z * sigma)


def calculate_parametric_es(returns, confidence):
    """Compute parametric ES assuming normally distributed returns.

    ES = -(mu - sigma * phi(z) / (1 - confidence)).
    Returns ES as a positive loss fraction.
    """
    mu, sigma = estimate_distribution(returns)
    z = float(stats.norm.ppf(confidence))
    alpha = 1.0 - confidence
    return -(mu - sigma * float(stats.norm.pdf(z)) / alpha)

In [10]:
var_pct = calculate_parametric_var(daily_returns, CONFIDENCE_LEVEL)
es_pct = calculate_parametric_es(daily_returns, CONFIDENCE_LEVEL)
print(f"1-Day VaR: {var_pct:.4f} ({var_pct*100:.2f}%)")
print(f"1-Day ES:  {es_pct:.4f} ({es_pct*100:.2f}%)")

1-Day VaR: 0.0417 (4.17%)
1-Day ES:  0.0481 (4.81%)


## 6. Orchestration of the Workflow

In [11]:
def parametric_var_es_pipeline(ticker, confidence, lookback, n_days, portfolio_value, end_date=None):
    """Run the full parametric VaR workflow.

    Fetches data, estimates distribution parameters, computes 1-day VaR/ES
    using the normal model, and scales results to the n-day horizon.
    Returns dollar VaR/ES along with the underlying data.
    """
    # 1. Fetch data and compute returns
    prices = fetch_prices(ticker, lookback, end_date)
    daily_returns = compute_returns(prices, kind="log")
    mu, sigma = estimate_distribution(daily_returns)

    # 2. Calculate 1-day VaR and ES
    var_1d_pct = calculate_parametric_var(daily_returns, confidence)
    es_1d_pct = calculate_parametric_es(daily_returns, confidence)
    var_1d = var_1d_pct * portfolio_value
    es_1d = es_1d_pct * portfolio_value

    # 3. Scale to N-day horizon
    scaling_factor = np.sqrt(n_days)
    var_nd = var_1d * scaling_factor
    es_nd = es_1d * scaling_factor

    return {
        "var_1d": var_1d,
        "var_nd": var_nd,
        "es_1d": es_1d,
        "es_nd": es_nd,
        "prices": prices,
        "daily_returns": daily_returns,
        "mu": mu,
        "sigma": sigma,
    }

In [12]:
results = parametric_var_es_pipeline(
    ticker=TICKER,
    confidence=CONFIDENCE_LEVEL,
    lookback=LOOKBACK_DAYS,
    n_days=HORIZON_DAYS,
    portfolio_value=PORTFOLIO_VALUE)

In [13]:
print("=" * 60)
print(f"PARAMETRIC VaR ANALYSIS SUMMARY - {TICKER}")
print("=" * 60)
print(f"VaR Date: {datetime.now().strftime('%Y-%m-%d')}")
print(f"Portfolio Value: ${PORTFOLIO_VALUE:,}")
print(f"Confidence Level: {CONFIDENCE_LEVEL:.0%}")
print(f"Time Horizon: {HORIZON_DAYS} days")
print(f"Historical Period: {LOOKBACK_DAYS} trading days")
print()
print("DISTRIBUTION PARAMETERS:")
print('-'*60)
print(f"  Mean daily return (mu):    {results['mu']:.6f}")
print(f"  Std dev (sigma):           {results['sigma']:.6f}")
print()
print("VaR METRICS:")
print('-'*60)
print(f"  {HORIZON_DAYS}-Day VaR: ${results['var_nd']:,.2f} ({results['var_nd']/PORTFOLIO_VALUE*100:.2f}%)")
print(f"  {HORIZON_DAYS}-Day ES:  ${results['es_nd']:,.2f} ({results['es_nd']/PORTFOLIO_VALUE*100:.2f}%)")
print()

PARAMETRIC VaR ANALYSIS SUMMARY - GOOG
VaR Date: 2026-03-28
Portfolio Value: $1,000,000
Confidence Level: 99%
Time Horizon: 10 days
Historical Period: 251 trading days

DISTRIBUTION PARAMETERS:
------------------------------------------------------------
  Mean daily return (mu):    0.002162
  Std dev (sigma):           0.018841

VaR METRICS:
------------------------------------------------------------
  10-Day VaR: $131,766.21 (13.18%)
  10-Day ES:  $151,955.80 (15.20%)



## 7. Assumption Tests

Parametric VaR rests on specific assumptions. The tests below verify that:
1. Returns are assumed to be **normally distributed**.
2. VaR is computed from **mean and standard deviation**.
3. **Tail risk is driven purely by volatility**.

### Test 1 – Normality of Returns

Parametric VaR assumes that daily returns follow a normal distribution.  
We apply two statistical tests to evaluate this assumption:

| Test | Purpose | Best for |
| --- | --- | --- |
| **Shapiro-Wilk** | General normality test (primary) | Small-to-medium samples (~250) |
| **Jarque-Bera** | Tests skewness & kurtosis specifically | Diagnosing *how* normality fails |

A **p-value > 0.05** means we cannot reject the null hypothesis of normality
at the 5% significance level. The Shapiro-Wilk result drives the overall
conclusion; Jarque-Bera provides supporting detail.

In [14]:
def test_normality(returns):
    """Test whether returns follow a normal distribution.

    Runs both the Shapiro-Wilk test (primary, suitable for n ~ 250)
    and the Jarque-Bera test (supporting, focuses on skewness and
    kurtosis).  Returns a dict with all test statistics.
    """
    # Shapiro-Wilk (primary)
    sw_stat, sw_p = stats.shapiro(returns)

    # Jarque-Bera (supporting diagnostic)
    jb_stat, jb_p = stats.jarque_bera(returns)

    # Descriptive shape statistics
    skewness = returns.skew()
    excess_kurtosis = returns.kurtosis()  # normal = 0

    return {
        "sw_stat": sw_stat,
        "sw_pvalue": sw_p,
        "jb_stat": jb_stat,
        "jb_pvalue": jb_p,
        "skewness": skewness,
        "excess_kurtosis": excess_kurtosis,
        "n_obs": len(returns),
        "is_normal": sw_p > 0.05
    }

In [15]:
normality = test_normality(daily_returns)

print("=" * 60)
print("TEST 1: Normality of Returns")
print("=" * 60)
print(f"  Sample size:       {normality['n_obs']}")
print()
print("SHAPE STATISTICS:")
print("-" * 60)
print(f"  Skewness:          {normality['skewness']:.4f}  (normal ~ 0)")
print(f"  Excess kurtosis:   {normality['excess_kurtosis']:.4f}  (normal ~ 0)")
print()
print("PRIMARY TEST  –  Shapiro-Wilk:")
print("-" * 60)
print(f"  Statistic:         {normality['sw_stat']:.6f}")
print(f"  p-value:           {normality['sw_pvalue']:.6f}")
print()
print("SUPPORTING TEST  –  Jarque-Bera:")
print("-" * 60)
print(f"  Statistic:         {normality['jb_stat']:.4f}")
print(f"  p-value:           {normality['jb_pvalue']:.6f}")
print()
print("CONCLUSION (based on Shapiro-Wilk):")
print("=" * 60)
if normality["is_normal"]:
    print("  PASS - Cannot reject normality at 5% significance.")
    print("  The normal-distribution assumption is reasonable for this sample.")
    print("  Data is considered normal as per Shapiro-Wilk test.")
else:
    print("  WARNING - Normality rejected (Shapiro-Wilk p < 0.05).")
    print("  Returns may exhibit fat tails or skew; Parametric VaR could underestimate true tail risk.")
    print("  Data is NOT normal as per Shapiro-Wilk test.")

TEST 1: Normality of Returns
  Sample size:       250

SHAPE STATISTICS:
------------------------------------------------------------
  Skewness:          0.5195  (normal ~ 0)
  Excess kurtosis:   4.2984  (normal ~ 0)

PRIMARY TEST  –  Shapiro-Wilk:
------------------------------------------------------------
  Statistic:         0.949339
  p-value:           0.000000

SUPPORTING TEST  –  Jarque-Bera:
------------------------------------------------------------
  Statistic:         193.9030
  p-value:           0.000000

CONCLUSION (based on Shapiro-Wilk):
  WARNING - Normality rejected (Shapiro-Wilk p < 0.05).
  Returns may exhibit fat tails or skew; Parametric VaR could underestimate true tail risk.
  Data is NOT normal as per Shapiro-Wilk test.
